In [15]:
from model.model import PoseModel
import torch
from data.dataset import LinemodDataset
import cv2
import numpy as np

from utils.predictions import estimate_pose
from utils.visualization import show_all

# Loading the model

In [16]:
checkpoint_path="checkpoints/best.pth"

In [17]:
model = PoseModel(num_objects=15).to("cuda")

    # Carica checkpoint
checkpoint = torch.load(checkpoint_path, map_location="cuda")

    # Carica pesi
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

PoseModel(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_

# Loading the dataset

In [18]:
dataset_root = "datasets/Linemod_preprocessed/"

val_dataset = LinemodDataset(dataset_root, split='test')



# Making a prediction and visualizing the corresponding 3d bb

In [ ]:
sample_id = np.random.randint(len(val_dataset))
sample_id=20
pred_r, pred_t, _ = estimate_pose(model, val_dataset, sample_id)


show_all(val_dataset[sample_id], pred_r,pred_t)

In [20]:
import os

In [21]:
def get_3d_model_path(dataset_root, obj_id):
    print(obj_id)
    return os.path.join(dataset_root, 'models', f"obj_{obj_id+1:02d}.ply")

In [22]:
from metrics.add import compute_add

In [23]:
sample = val_dataset[sample_id]

In [24]:
model_path = get_3d_model_path(dataset_root,sample['obj_id'])

tensor(7)


In [25]:
compute_add(model_path, sample['rotation'], sample['translation'], pred_r, pred_t)

np.float64(35.726149475953775)

In [26]:
from metrics.add import compute_add_percent

In [27]:
compute_add_percent(model_path, sample['rotation'], sample['translation'], pred_r,pred_t)

(np.float64(35.726149475953775),
 np.float64(88.79281290769337),
 np.float64(318.7789155449954))

In [28]:
add_perc_cum = 0
for sample_id in range(len(val_dataset)):
    sample = val_dataset[sample_id]
    pred_r, pred_t, _ = estimate_pose(model, val_dataset, sample_id)
    model_path = get_3d_model_path(dataset_root, sample['obj_id'])
    add_abs, add_perc, _ = compute_add_percent(model_path, sample['rotation'], sample['translation'], pred_r,pred_t)
    add_perc_cum = add_perc_cum + add_perc
    curr_perc = add_perc_cum / (sample_id + 1)
    print(f"Current add: {curr_perc}%")
    
    

tensor(4)
Current add: 87.53926439087945%
tensor(0)
Current add: 92.0688686762611%
tensor(4)
Current add: 90.21898468585853%
tensor(12)
Current add: 89.89274415434528%
tensor(0)
Current add: 80.60439011616062%
tensor(0)
Current add: 79.26483767992194%
tensor(13)
Current add: 79.55782895167397%
tensor(13)
Current add: 74.08749090690928%
tensor(12)
Current add: 65.85554747280825%
tensor(14)
Current add: 63.791987401520274%
tensor(7)
Current add: 66.06478972026328%
tensor(13)
Current add: 66.32208455922337%
tensor(8)
Current add: 68.13789809903503%
tensor(3)
Current add: 69.84166351724788%
tensor(9)
Current add: 65.18555261609802%
tensor(0)
Current add: 61.44701172297688%
tensor(0)
Current add: 57.8324816216253%
tensor(4)
Current add: 59.68527770660062%
tensor(8)
Current add: 58.06175330463015%
tensor(10)
Current add: 59.24109126156959%
tensor(10)
Current add: 56.420086915780566%
tensor(11)
Current add: 53.85553751051781%
tensor(8)
Current add: 55.270139499299496%
tensor(3)
Current add: 5

KeyboardInterrupt: 

In [ ]:
add_perc_cum / len(val_dataset)